In [1]:
import os
import json
import shutil
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
NUM_TOPICS = 20  # TODO: this value does not affect anything

NUM_ITERATIONS = 5
NUM_TOP_TOKENS = 20
NUM_TRAINS = 20

In [6]:
BERTOPIC_FOLDER_PATH = '/data_mil/shared/CompressaAI/BERTopic'

In [7]:
! ls $BERTOPIC_FOLDER_PATH

results  results50


In [8]:
! ls $BERTOPIC_FOLDER_PATH/results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [9]:
RESULTS_FOLDER_PATH = os.path.join(BERTOPIC_FOLDER_PATH, 'results', 'ruwikigood')

In [10]:
RESULTS_FOLDER_PATH

'/data_mil/shared/CompressaAI/BERTopic/results/ruwikigood'

In [11]:
! ls results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [12]:
SAVE_FOLDER = os.path.join('results', 'ruwikigood')

In [13]:
SAVE_FOLDER

'results/ruwikigood'

In [14]:
! ls $SAVE_FOLDER

ablation_study			     iterative2_1000000000
decorrelation.json		     iterative2_10000000000
iterative_1000000		     iterative2_100000000000
iterative_10000000		     iterative2_100000000000.json
iterative_100000000		     iterative2_10000000000.json
iterative_100000000_unfinished.json  iterative2_1000000000.json
_iterative_10000000.json	     lda.json
iterative_10000000.json		     plsa.json
_iterative_1000000.json		     sparse.json
iterative_1000000.json		     tless.json


In [15]:
! ls $RESULTS_FOLDER_PATH

0  1  10  11  12  13  14  15  16  17  18  19  2  3  4  5  6  7	8  9


In [16]:
! ls $RESULTS_FOLDER_PATH/0

dataset.csv  phi.csv  top_words.json


In [17]:
dataset = Dataset(
    f'{RESULTS_FOLDER_PATH}/0/dataset.csv',
)

dataset.get_possible_modalities()

{'@lemmatized'}

In [18]:
MAIN_MODALITY = '@lemmatized'

In [19]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [20]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 1min 32s, sys: 32.6 s, total: 2min 4s
Wall time: 2min 3s


In [21]:
co_occurences.shape

(264943, 264943)

In [22]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [23]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [24]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [25]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9

    def __init__(self, name: str, topic_names: List[str], parent_model=None, parent_phi=None, words=None):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._words = words
        self._word_indices = None
        self._parent_model = parent_model
        self._parent_phi = parent_phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)

        if self._parent_phi is not None:
            parent_phi = self._parent_phi
            vals = parent_phi.values
        else:
            assert False

            parent_phi = self._parent_model.get_phi()
            vals = parent_phi.values[:, self._topic_indices]

        if self._word_indices is None:
            assert vals.shape[0] == rwt.shape[0], (vals.shape[0], rwt.shape[0])
        else:
            assert vals.shape[0] == len(self._word_indices)

        assert vals.shape[1] == len(self._topic_indices)

        if self._word_indices is None:
            rwt[:, self._topic_indices] += vals
        else:
            # print(len(self._word_indices), len(self._topic_indices), rwt.shape, vals.shape)
            # print(self._word_indices[:3], self._topic_indices[:3])
            # print(rwt[np.ix_(self._word_indices, self._topic_indices)].shape, vals.shape)

            # https://github.com/numpy/numpy/issues/5574
            # https://github.com/numpy/numpy/issues/13255
            rwt[np.ix_(self._word_indices, self._topic_indices)] += vals

            # print(np.ix_(self._word_indices, self._topic_indices)[:3])

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

        if self._words is not None:
            self._word_indices = [
                # phi.index.get_loc(w) for w in self._words
                int(phi.index.get_loc(w)) for w in self._words
            ]

            # print(f'!!! Word indices: {self._word_indices}')

In [26]:
def fit_and_compute_scores(model, dataset, target_topic_indices=None, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account

    if target_topic_indices is None:
        target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()

    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [27]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [28]:
TOPIC_INDICES = list(range(NUM_TOPICS))

In [29]:
TOPIC_INDICES

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [30]:
phi0 = pd.read_csv(f'{RESULTS_FOLDER_PATH}/0/phi.csv', index_col=0)

In [31]:
phi0.head()

,background_1,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15
aa,0.000019,0.000032,0.000065,0.0,0.00000,0.000016,0.0,0.000019,0.0,0.0,0.0,0.0,0.007627,0.0,0.0,0.0,0.0
aaa,0.000000,0.000072,0.000011,0.0,0.00003,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
aaaaa,0.000003,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
aaaaabbbbbabbbaabbababbaaababaab,0.000003,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
aaaab,0.000004,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [32]:
phi0.set_index([[MAIN_MODALITY] * len(phi0.index), phi0.index], inplace=True)

In [124]:
phi0.head()

background_1   topic_0  \
@lemmatized aa                                    0.000019  0.000032   
            aaa                                   0.000000  0.000072   
            aaaaa                                 0.000003  0.000000   
            aaaaabbbbbabbbaabbababbaaababaab      0.000003  0.000000   
            aaaab                                 0.000004  0.000000   

                                               topic_1  topic_2  topic_3  \
@lemmatized aa                                0.000065      0.0  0.00000   
            aaa                               0.000011      0.0  0.00003   
            aaaaa                             0.000000      0.0  0.00000   
            aaaaabbbbbabbbaabbababbaaababaab  0.000000      0.0  0.00000   
            aaaab                             0.000000      0.0  0.00000   

                                               topic_4  topic_5   topic_6  \
@lemmatized aa                                0.000016      0.0  0.000019   
            aaa                               0.000000      0.0  0.000000   
            aaaaa                             0.000000      0.0  0.000000   
            aaaaabbbbbabbbaabbababbaaababaab  0.000000      0.0  0.000000   
            aaaab                             0.000000      0.0  0.000000   

                                              topic_7  topic_8  topic_9  \
@lemmatized aa                                    0.0      0.0      0.0   
            aaa                                   0.0      0.0      0.0   
            aaaaa                                 0.0      0.0      0.0   
            aaaaabbbbbabbbaabbababbaaababaab      0.0      0.0      0.0   
            aaaab                                 0.0      0.0      0.0   

                                              topic_10  topic_11  topic_12  \
@lemmatized aa                                     0.0  0.007627       0.0   
            aaa                                    0.0  0.000000       0.0   
            aaaaa                                  0.0  0.000000       0.0   
            aaaaabbbbbabbbaabbababbaaababaab       0.0  0.000000       0.0   
            aaaab                                  0.0  0.000000       0.0   

                                              topic_13  topic_14  topic_15  
@lemmatized aa                                     0.0       0.0       0.0  
            aaa                                    0.0       0.0       0.0  
            aaaaa                                  0.0       0.0       0.0  
            aaaaabbbbbabbbaabbababbaaababaab       0.0       0.0       0.0  
            aaaab                                  0.0       0.0       0.0

In [125]:
proper_names = [f'topic_{i}' for i in range(phi0.shape[1])]

phi0.rename(
    columns={old_name: new_name for old_name, new_name in zip(phi0.columns, proper_names)},
    inplace=True,
)

In [126]:
phi0.head()

topic_0   topic_1   topic_2  \
@lemmatized aa                                0.000019  0.000032  0.000065   
            aaa                               0.000000  0.000072  0.000011   
            aaaaa                             0.000003  0.000000  0.000000   
            aaaaabbbbbabbbaabbababbaaababaab  0.000003  0.000000  0.000000   
            aaaab                             0.000004  0.000000  0.000000   

                                              topic_3  topic_4   topic_5  \
@lemmatized aa                                    0.0  0.00000  0.000016   
            aaa                                   0.0  0.00003  0.000000   
            aaaaa                                 0.0  0.00000  0.000000   
            aaaaabbbbbabbbaabbababbaaababaab      0.0  0.00000  0.000000   
            aaaab                                 0.0  0.00000  0.000000   

                                              topic_6   topic_7  topic_8  \
@lemmatized aa                                    0.0  0.000019      0.0   
            aaa                                   0.0  0.000000      0.0   
            aaaaa                                 0.0  0.000000      0.0   
            aaaaabbbbbabbbaabbababbaaababaab      0.0  0.000000      0.0   
            aaaab                                 0.0  0.000000      0.0   

                                              topic_9  topic_10  topic_11  \
@lemmatized aa                                    0.0       0.0       0.0   
            aaa                                   0.0       0.0       0.0   
            aaaaa                                 0.0       0.0       0.0   
            aaaaabbbbbabbbaabbababbaaababaab      0.0       0.0       0.0   
            aaaab                                 0.0       0.0       0.0   

                                              topic_12  topic_13  topic_14  \
@lemmatized aa                                0.007627       0.0       0.0   
            aaa                               0.000000       0.0       0.0   
            aaaaa                             0.000000       0.0       0.0   
            aaaaabbbbbabbbaabbababbaaababaab  0.000000       0.0       0.0   
            aaaab                             0.000000       0.0       0.0   

                                              topic_15  topic_16  
@lemmatized aa                                     0.0       0.0  
            aaa                                    0.0       0.0  
            aaaaa                                  0.0       0.0  
            aaaaabbbbbabbbaabbababbaaababaab       0.0       0.0  
            aaaab                                  0.0       0.0

In [35]:
DIFF_THRESHOLD = 2

In [185]:
def check_top_words(phi, top_words):
    diffs = []

    for i, (t, topic_top_words) in enumerate(top_words.items()):
        if t == 'background_1':
            print(f'Skipping background topic: {t}.')

            # Sometimes it is here...
            # assert not topic_top_words  # TODO: BERTopic did not give -1 topic for GoodRuWiki...
                                        #   So, we have (model phi Vs. top words)
                                        #                     -            "background_1"
                                        #               "background_1" <-> "topic_0"
                                        #               "topic_1"      <-> "topic_2"
                                        #                              ...
                                        #               "topic_n-1"    <-> "topic_n"
                                        #               "topic_n"      <->     - (lost top words)

            continue

        print(t)
        # print(top_words)
    
        top_phi = set(phi[t].sort_values(ascending=False)[:NUM_TOP_TOKENS].index.get_level_values(1))
        top_bt = set([p[0] for p in topic_top_words])
    
        if top_phi == top_bt:
            diffs.append(
                {
                    'total': 0,
                    'lost_bt': 0,
                    'lost_model': 0,
                }
            )
        else:
            diff1 = top_phi.difference(top_bt)
            diff2 = top_bt.difference(top_phi)
    
            print('  WTF:', diff1, diff2)
    
            if len(diff1) > DIFF_THRESHOLD:
                print(f'  WTF?!?!?', len(diff1))
    
            if len(diff2) > DIFF_THRESHOLD:
                print(f'  WTF?!?!?', len(diff2))

            diffs.append(
                {
                    'total': len(diff1 | diff2),
                    'lost_bt': len(diff2),
                    'lost_model': len(diff1),
                }
            )

    return diffs

In [210]:
[phi.columns[-1]] + list(phi.columns[:-1])

['topic_19',
 'topic_0',
 'topic_1',
 'topic_2',
 'topic_3',
 'topic_4',
 'topic_5',
 'topic_6',
 'topic_7',
 'topic_8',
 'topic_9',
 'topic_10',
 'topic_11',
 'topic_12',
 'topic_13',
 'topic_14',
 'topic_15',
 'topic_16',
 'topic_17',
 'topic_18']

In [214]:
def init_model_and_phi(num_topics, dataset, phi0):
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=num_topics,
        seed=0,
    )
    
    phi = model.get_phi()
    
    # assert phi.shape[1] == phi0.shape[1] - 1
    assert phi.shape[1] == num_topics
    # assert phi.shape[1] in [phi0.shape[1], phi0.shape[1] - 1]
    # assert phi.shape[1] in [phi0.shape[1], phi0.shape[1] + 1]  # TODO: GoodRuWiki specific fix
    assert phi.shape[1] in [
        phi0.shape[1],      # just save all
        phi0.shape[1] + 1,  # all plus bcg
        phi0.shape[1] - 1   # all minus bcg
    ]

    common_words = list(set(phi.index).intersection(phi0.index))
    
    phi.loc[:, :] = 1

    if phi.shape[1] == phi0.shape[1]:  # phi0 has bcg or does not
        target_topics = phi.columns
        source_topics = phi0.columns

        if 'back' in source_topics[0]:
            target_topics = [target_topics[-1]] + list(target_topics[:-1])  # bcg -- last

        # assert set(phi.columns) == set(phi0.columns)

    elif phi.shape[1] == phi0.shape[1] - 1:  # no bcg
        target_topics = phi.columns
        source_topics = [t for t in phi0.columns if 'back' not in t]

        assert list(target_topics) == list(source_topics), (target_topics, source_topics)

    elif phi.shape[1] == phi0.shape[1] + 1:  # with bcg but phi0 does not have bcg (TODO: GoodRuWiki specific)
        target_topics = phi0.columns
        source_topics = phi0.columns

    assert len(target_topics) == len(source_topics), (target_topics, source_topics)

    # print(common_words, phi.index, phi0.index)

    phi.loc[:, target_topics] = 0
    phi.loc[common_words, target_topics] = phi0.loc[common_words, source_topics]  # Fix for GoodRuWiki (but seems would be OK for all)

    diff = set(phi0.columns).symmetric_difference(set(phi.columns))

    # assert diff == {'background_1'} or diff == {'background_1', f'topic_{len(phi.columns) - 1}'}
    # assert diff == set() or diff == {f'topic_{len(phi.columns) - 1}'}  # TODO: GoodRuWiki specific
    assert diff == set() or diff == {f'topic_{len(phi.columns) - 1}'} or diff == {'background_1'} or diff == {'background_1', f'topic_{len(phi.columns) - 1}'}, (
        diff, phi0.columns, phi.columns
    )

    phi = phi / phi.sum(axis=0)

    return model, phi, common_words

In [215]:
not not 1

True

In [216]:
results = []

is_results_loaded = False

save_file_path = os.path.join(
    SAVE_FOLDER, 'bertopic.json'
)

if os.path.isfile(save_file_path):
    results = json.loads(
        open(save_file_path).read()
    )
    is_results_loaded = True


singles_save_folder = os.path.join(
    SAVE_FOLDER, 'bertopic'
)

os.makedirs(singles_save_folder, exist_ok=True)


for seed in range(NUM_TRAINS):
    if is_results_loaded:
        continue

    print(seed)

    current_save_file_path = os.path.join(
        singles_save_folder, f'bertopic_{seed}.json'
    )

    if os.path.isfile(current_save_file_path):
        print(f'Already computed results: {current_save_file_path}. Loading and skipping...')

        result = json.loads(
            open(current_save_file_path).read()
        )
        results.append(result)

        continue

    seed_load_folder = os.path.join(RESULTS_FOLDER_PATH, str(seed))
    files = [f for f in os.listdir(seed_load_folder) if os.path.isfile(f'{seed_load_folder}/{f}')]
    
    assert len(files) == 3

    dataset = Dataset(
        f'{seed_load_folder}/dataset.csv',
    )

    with open(f'{seed_load_folder}/top_words.json', 'r') as f:
        top_words = json.loads(f.read())

    has_bcg = not not top_words['background_1']
    
    phi0 = pd.read_csv(f'{seed_load_folder}/phi.csv', index_col=0)
    phi0.set_index([[MAIN_MODALITY] * len(phi0.index), phi0.index], inplace=True)

    if not has_bcg:
        # *** GoodRuWiki specific code ***
        proper_names = [f'topic_{i}' for i in range(phi0.shape[1])]
        phi0.rename(
            columns={old_name: new_name for old_name, new_name in zip(phi0.columns, proper_names)},
            inplace=True,
        )
        # *** GoodRuWiki specific code ***

        num_specific_topics = phi0.shape[1]  # - 1
    else:
        assert all('back' not in t for t in phi0.columns[1:])
        assert 'back' in phi0.columns[0]

        num_specific_topics = phi0.shape[1] - 1

    if not has_bcg:
        assert num_specific_topics == len(phi0.columns)  # len([t for t in phi0.columns if 'back' not in t])
    else:
        assert num_specific_topics == len([t for t in phi0.columns if 'back' not in t])

    print(f'Num model topics: {phi0.shape[1]}.')


    
    model, phi, common_phi_words = init_model_and_phi(
        num_topics=num_specific_topics + 1,  # background
        dataset=dataset, phi0=phi0
    )
    common_words = [
        w[1] for w in common_phi_words  # without modality
    ]
    common_words_phi0_indices = [
        # phi0.index.get_loc(w) for w in common_words
        int(phi0.index.get_locs(w)) for w in common_phi_words
    ]

    print('Check before fit:')
    diff_tops1 = check_top_words(phi, top_words)  # Whatever...

    if not has_bcg:
        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_phi=phi0.iloc[common_words_phi0_indices, :],  # TODO: GoodRuWiki specific change (no bcg) 
            words=common_words,
            topic_names=phi.columns[:-1],  # TODO: GoodRuWiki specific change (bcg topic -- last)
        )
    else:
        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_phi=phi0.iloc[common_words_phi0_indices, 1:],
            words=common_words,
            topic_names=phi.columns[:-1],  # TODO: GoodRuWiki specific change (bcg topic -- last)
        )

    result = fit_and_compute_scores(
        model, dataset,
        target_topic_indices=list(range(phi.shape[1])),
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )

    print('Check after fit:')
    diff_tops2 = check_top_words(phi, top_words)  # Whatever...

    assert diff_tops1 == diff_tops2

    fair_ppl_free = result['scores']['perplexity']

    del model, phi, fix_regularizer, result



    if has_bcg:
        model, phi, _ = init_model_and_phi(
            num_topics=num_specific_topics + 1,  # background
            dataset=dataset, phi0=phi0
        )
        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_phi=phi0.iloc[common_words_phi0_indices, :],  # Diff here
            words=common_words,
            topic_names=phi.columns,
        )
        result = fit_and_compute_scores(
            model, dataset,
            target_topic_indices=list(range(phi.shape[1])),
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
            }
        )
        
        fair_ppl_fix = result['scores']['perplexity']
    
        del model, phi, fix_regularizer, result
    else:
        fair_ppl_fix = None
    

    
    model, phi, _ = init_model_and_phi(
        num_topics=num_specific_topics,  # TODO: diff here
        dataset=dataset, phi0=phi0
    )

    if not has_bcg:
        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_phi=phi0.iloc[common_words_phi0_indices, :],  # TODO: diff here
            words=common_words,
            topic_names=phi.columns,
        )
    else:
        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_phi=phi0.iloc[common_words_phi0_indices, 1:],  # TODO: diff here
            words=common_words,
            topic_names=phi.columns,
        )

    result = fit_and_compute_scores(
        model, dataset,
        target_topic_indices=list(range(phi.shape[1])),  # TODO: not bad a fix
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )

    unfair_ppl_banklike = result['scores']['perplexity']


    
    result['scores']['fair_ppl_free'] = fair_ppl_free
    result['scores']['fair_ppl_fix'] = fair_ppl_fix
    result['scores']['unfair_ppl_banklike'] = unfair_ppl_banklike

    assert result['scores']['fair_ppl_free'] < result['scores']['unfair_ppl_banklike']
    # assert result['scores']['fair_ppl_fix'] < result['scores']['unfair_ppl_banklike']

    result['scores']['coherence_20'] = float(result['scores']['coherence_20'])
    result['stats'] = {
        'num_topics': phi0.shape[1],
        'num_common_words': len(common_words),
        'num_model_words': phi.shape[0],
        'num_bt_words': phi0.shape[0],
        'top_diffs': diff_tops2,
    }

    results.append(result)

    print(result['scores'])
    print(result['stats'])

    dumped_result = json.dumps(
        results[-1], indent=4
    )

    with open(current_save_file_path, 'w') as f:
        f.write(dumped_result)

    del model, phi, fix_regularizer


with open(save_file_path, 'w') as f:
    f.write(
        json.dumps(
            results, indent=4
        )
    )

0
Already computed results: results/ruwikigood/bertopic/bertopic_0.json. Loading and skipping...
1
Already computed results: results/ruwikigood/bertopic/bertopic_1.json. Loading and skipping...
2
Already computed results: results/ruwikigood/bertopic/bertopic_2.json. Loading and skipping...
3
Already computed results: results/ruwikigood/bertopic/bertopic_3.json. Loading and skipping...
4
Already computed results: results/ruwikigood/bertopic/bertopic_4.json. Loading and skipping...
5
Already computed results: results/ruwikigood/bertopic/bertopic_5.json. Loading and skipping...
6
Already computed results: results/ruwikigood/bertopic/bertopic_6.json. Loading and skipping...
7
Already computed results: results/ruwikigood/bertopic/bertopic_7.json. Loading and skipping...
8
Already computed results: results/ruwikigood/bertopic/bertopic_8.json. Loading and skipping...
9
Num model topics: 20.


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



Check before fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16
topic_17
topic_18
{'fix': <__main__.FastFixPhiRegularizer object at 0x7efdd0c84220>}
Check after fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16
topic_17
topic_18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7efdd0c84a00>}


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7efdd26bad60>}
{'perplexity': 6006.9013671875, 'coherence_20': 1.1946819355160077, 'diversity_euclidean': 0.04867373147741286, 'diversity_jensenshannon': 0.6568019697282852, 'diversity_hellinger': 0.7610876702424425, 'diversity_cosine': 0.8225186116846672, 'fair_ppl_free': 4053.443115234375, 'fair_ppl_fix': None, 'unfair_ppl_banklike': 6006.9013671875}
{'num_topics': 20, 'num_common_words': 264923, 'num_model_words': 264943, 'num_bt_words': 264925, 'top_diffs': [{'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'los

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



Check before fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16
{'fix': <__main__.FastFixPhiRegularizer object at 0x7efe118e41f0>}
Check after fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7efe697d4670>}
{'perplexity': 6198.82470703125, 'coherence_20': 1.167396157915651, 'diversity_euclidean': 0.04846391519781991, 'diversity_jensenshannon': 0.6637423358186256, 'diversity_hellinger': 0.7705792274710391, 'diversity_cosine': 0.8199637123306177, 'fair_ppl_free': 4194.4365234375, 'fair_ppl_fix': None, 'unfair_ppl_banklike': 6198.82470703125}
{'num_topics': 18, 'num_common_words': 264923, 'num_model_words': 264943, 'num_bt_words': 264925, 'top_diffs': [{'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost

In [218]:
phi.columns

NameError: name 'phi' is not defined

In [219]:
1-201

1

In [ ]:
! cat $RESULTS_FOLDER_PATH/9/top_words.json

In [177]:
NUM_TOPICS

20

In [176]:
phi.shape

(264943, 17)

In [149]:
phi.head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
modality    token                                                           
@lemmatized gdi          2.563433e-07      0.0      0.0      0.0      0.0   
            egosoft      2.563433e-07      0.0      0.0      0.0      0.0   
            ogame        4.866823e-07      0.0      0.0      0.0      0.0   
            матёс        2.563433e-07      0.0      0.0      0.0      0.0   
            шильцевский  2.563433e-07      0.0      0.0      0.0      0.0   

                         topic_5  topic_6  topic_7  topic_8  topic_9  \
modality    token                                                      
@lemmatized gdi              0.0      0.0      0.0      0.0      0.0   
            egosoft          0.0      0.0      0.0      0.0      0.0   
            ogame            0.0      0.0      0.0      0.0      0.0   
            матёс            0.0      0.0      0.0      0.0      0.0   
            шильцевский      0.0      0.0      0.0      0.0      0.0   

                         topic_10  topic_11  topic_12  topic_13  topic_14  \
modality    token                                                           
@lemmatized gdi               0.0       0.0       0.0       0.0       0.0   
            egosoft           0.0       0.0       0.0       0.0       0.0   
            ogame             0.0       0.0       0.0       0.0       0.0   
            матёс             0.0       0.0       0.0       0.0       0.0   
            шильцевский       0.0       0.0       0.0       0.0       0.0   

                         topic_15  topic_16  topic_17  
modality    token                                      
@lemmatized gdi               0.0       0.0  0.000004  
            egosoft           0.0       0.0  0.000004  
            ogame             0.0       0.0  0.000004  
            матёс             0.0       0.0  0.000004  
            шильцевский       0.0       0.0  0.000004

In [163]:
phi['topic_16'].sort_values(ascending=False)[:20]

modality     token   
@lemmatized  покемон     0.043134
             игра        0.017567
             pokemon     0.015402
             nintendo    0.009551
             игрок       0.007089
             пикача      0.006280
             серия       0.006174
             тренер      0.005622
             game        0.005394
             эш          0.004943
             red         0.004613
             аниме       0.004124
             blue        0.003957
             сериал      0.003954
             бой         0.003804
             boy         0.003653
             регион      0.003261
             ruby        0.003222
             sapphire    0.003204
             игровой     0.002989
Name: topic_16, dtype: float64

In [168]:
phi0['topic_16'].sort_values(ascending=False)[:20]

@lemmatized  покемон     0.268106
             игра        0.109189
             pokemon     0.095733
             nintendo    0.059366
             игрок       0.044063
             пикача      0.039032
             серия       0.038374
             тренер      0.034944
             game        0.033527
             эш          0.030722
             red         0.028671
             аниме       0.025636
             blue        0.024596
             сериал      0.024576
             бой         0.023642
             boy         0.022706
             регион      0.020267
             ruby        0.020025
             sapphire    0.019915
             игровой     0.018577
Name: topic_16, dtype: float64

In [167]:
phi0['topic_15'].sort_values(ascending=False)[:20]

@lemmatized  малдёр           0.138623
             эпизод           0.116320
             агент            0.049091
             person           0.047593
             сериал           0.041120
             тумс             0.037865
             секретный        0.036518
             картер           0.029176
             фбр              0.027501
             материал         0.027373
             серия            0.025630
             fox              0.024078
             сцена            0.023637
             съёмка           0.021304
             сценарий         0.019872
             малдрать         0.019669
             домохозяйство    0.019580
             премьера         0.018778
             сезон            0.018749
             расследовать     0.017247
Name: topic_15, dtype: float64

In [166]:
top_words['topic_15']

[['малдёр', 0.1386228369280603],
 ['эпизод', 0.11631993906668163],
 ['агент', 0.049090525093543905],
 ['person', 0.047593315648508897],
 ['сериал', 0.04111972980469913],
 ['тумс', 0.03786522487257441],
 ['секретный', 0.03651821212407736],
 ['картер', 0.02917647118432673],
 ['фбр', 0.027500518993828337],
 ['материал', 0.027373011451345342],
 ['серия', 0.025629757135196673],
 ['fox', 0.024077945281126726],
 ['сцена', 0.023637398476370437],
 ['съёмка', 0.021304282715615113],
 ['сценарий', 0.019872107173757463],
 ['малдрать', 0.019668540165023844],
 ['домохозяйство', 0.019580396028063024],
 ['премьера', 0.01877765221036062],
 ['сезон', 0.018749264086400926],
 ['расследовать', 0.01724706793607795]]

In [111]:
phi.shape

(264943, 17)

In [112]:
phi0.shape

(264925, 17)

In [66]:
phi.shape, len(top_words)

((264943, 17), 17)

In [46]:
check_top_words(phi, top_words)

Skipping background topic: background_1.
topic_0
  WTF: {'птица', 'клетка', 'вид', 'самка', 'ген', 'длина', 'белка', 'род', 'рнк', 'некоторый', 'акула', 'самец', 'белок', 'днк', 'растение'} {'большой', 'время', 'армия', 'книга', 'век', 'работа', 'находиться', 'стать', 'язык', 'город', 'война', 'часть', 'получить', 'человек', 'имя'}
  WTF?!?!? 15
  WTF?!?!? 15
topic_1
  WTF: {'политический', 'римский', 'претор', 'цезарь', 'сципион', 'гракх', 'марка', 'квинта', 'рим', 'сын', 'красс', 'война', 'сенат', 'источник', 'армия', 'консул'} {'хороший', 'игровой', 'версия', 'актёр', 'игрок', 'герой', 'серия', 'фильм', 'сериал', 'сцена', 'игра', 'роль', 'эпизод', 'режиссёр', 'картина', 'персонаж'}
  WTF?!?!? 16
  WTF?!?!? 16
topic_2
  WTF: {'выиграть', 'счёт', 'команда', 'игрок', 'гол', 'сезон', 'игра', 'клуб', 'стать', 'матч', 'тренер', 'турнир', 'кубок', 'сборный', 'забить', 'победа', 'лига', 'чемпионат'} {'птица', 'клетка', 'белок', 'вид', 'мочь', 'самка', 'ген', 'иметь', 'длина', 'белка', 'род'

[{'total': 30, 'lost_bt': 15, 'lost_model': 15},
 {'total': 32, 'lost_bt': 16, 'lost_model': 16},
 {'total': 36, 'lost_bt': 18, 'lost_model': 18},
 {'total': 32, 'lost_bt': 16, 'lost_model': 16},
 {'total': 36, 'lost_bt': 18, 'lost_model': 18},
 {'total': 36, 'lost_bt': 18, 'lost_model': 18},
 {'total': 38, 'lost_bt': 19, 'lost_model': 19},
 {'total': 38, 'lost_bt': 19, 'lost_model': 19},
 {'total': 38, 'lost_bt': 19, 'lost_model': 19},
 {'total': 38, 'lost_bt': 19, 'lost_model': 19},
 {'total': 38, 'lost_bt': 19, 'lost_model': 19},
 {'total': 38, 'lost_bt': 19, 'lost_model': 19},
 {'total': 40, 'lost_bt': 20, 'lost_model': 20},
 {'total': 40, 'lost_bt': 20, 'lost_model': 20},
 {'total': 40, 'lost_bt': 20, 'lost_model': 20},
 {'total': 40, 'lost_bt': 20, 'lost_model': 20}]

In [59]:
! cat '/data_mil/shared/CompressaAI/BERTopic/results/ruwikigood/0/top_words.json'

{
    "background_1": false,
    "topic_0": [
        [
            "person",
            0.0422573407299807
        ],
        [
            "год",
            0.029529524372319878
        ],
        [
            "время",
            0.012102291687147673
        ],
        [
            "стать",
            0.010141119486329416
        ],
        [
            "город",
            0.009177191305420053
        ],
        [
            "человек",
            0.0085192620682621
        ],
        [
            "часть",
            0.008051808916342945
        ],
        [
            "век",
            0.007805910871055686
        ],
        [
            "имя",
            0.007420150137652238
        ],
        [
            "армия",
            0.007204607579826462
        ],
        [
            "являться",
            0.007019571092204588
        ],
        [
            "мочь",
            0.006865394327750571
        ],
        [
            "иметь",
            0.00684777818019

In [53]:
phi0.columns

Index(['background_1', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4',
       'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10',
       'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15'],
      dtype='object')

In [64]:
phi0['background_1'].sort_values(ascending=False)[:20]

@lemmatized  person        0.042257
             год           0.029530
             время         0.012102
             стать         0.010141
             город         0.009177
             человек       0.008519
             часть         0.008052
             век           0.007806
             имя           0.007420
             армия         0.007205
             являться      0.007020
             мочь          0.006865
             иметь         0.006848
             война         0.006680
             получить      0.006673
             работа        0.006518
             большой       0.006191
             находиться    0.005973
             язык          0.005945
             книга         0.005661
Name: background_1, dtype: float64

In [60]:
phi0['topic_0'].sort_values(ascending=False)[:20]

@lemmatized  person      0.044236
             игра        0.038738
             фильм       0.034461
             год         0.021441
             персонаж    0.018764
             роль        0.016915
             время       0.013180
             сцена       0.012968
             герой       0.012618
             актёр       0.012025
             серия       0.011883
             режиссёр    0.011813
             игрок       0.011805
             стать       0.011129
             сериал      0.011006
             картина     0.010805
             хороший     0.010353
             игровой     0.009919
             эпизод      0.009715
             версия      0.009561
Name: topic_0, dtype: float64

In [61]:
phi0['topic_0'].sort_values(ascending=False)[:20]

@lemmatized  person      0.044236
             игра        0.038738
             фильм       0.034461
             год         0.021441
             персонаж    0.018764
             роль        0.016915
             время       0.013180
             сцена       0.012968
             герой       0.012618
             актёр       0.012025
             серия       0.011883
             режиссёр    0.011813
             игрок       0.011805
             стать       0.011129
             сериал      0.011006
             картина     0.010805
             хороший     0.010353
             игровой     0.009919
             эпизод      0.009715
             версия      0.009561
Name: topic_0, dtype: float64

In [62]:
phi['topic_0'].sort_values(ascending=False)[:20]

modality     token   
@lemmatized  person      0.007280
             игра        0.006375
             фильм       0.005671
             год         0.003528
             персонаж    0.003088
             роль        0.002784
             время       0.002169
             сцена       0.002134
             герой       0.002077
             актёр       0.001979
             серия       0.001956
             режиссёр    0.001944
             игрок       0.001943
             стать       0.001832
             сериал      0.001811
             картина     0.001778
             хороший     0.001704
             игровой     0.001632
             эпизод      0.001599
             версия      0.001573
Name: topic_0, dtype: float64

In [67]:
phi0['background_1'].sort_values(ascending=False)[:20]

@lemmatized  person        0.042257
             год           0.029530
             время         0.012102
             стать         0.010141
             город         0.009177
             человек       0.008519
             часть         0.008052
             век           0.007806
             имя           0.007420
             армия         0.007205
             являться      0.007020
             мочь          0.006865
             иметь         0.006848
             война         0.006680
             получить      0.006673
             работа        0.006518
             большой       0.006191
             находиться    0.005973
             язык          0.005945
             книга         0.005661
Name: background_1, dtype: float64

In [63]:
top_words['topic_0']

[['person', 0.0422573407299807],
 ['год', 0.029529524372319878],
 ['время', 0.012102291687147673],
 ['стать', 0.010141119486329416],
 ['город', 0.009177191305420053],
 ['человек', 0.0085192620682621],
 ['часть', 0.008051808916342945],
 ['век', 0.007805910871055686],
 ['имя', 0.007420150137652238],
 ['армия', 0.007204607579826462],
 ['являться', 0.007019571092204588],
 ['мочь', 0.006865394327750571],
 ['иметь', 0.006847778180194606],
 ['война', 0.006679773966025352],
 ['получить', 0.006672689870839094],
 ['работа', 0.006517804032917204],
 ['большой', 0.006191131955561368],
 ['находиться', 0.005972721385662755],
 ['язык', 0.005945349787161841],
 ['книга', 0.005661034032538145]]

In [116]:
phi0['background_1'].sort_values(ascending=False)[:20]

@lemmatized  person        0.042257
             год           0.029530
             время         0.012102
             стать         0.010141
             город         0.009177
             человек       0.008519
             часть         0.008052
             век           0.007806
             имя           0.007420
             армия         0.007205
             являться      0.007020
             мочь          0.006865
             иметь         0.006848
             война         0.006680
             получить      0.006673
             работа        0.006518
             большой       0.006191
             находиться    0.005973
             язык          0.005945
             книга         0.005661
Name: background_1, dtype: float64

In [122]:
phi0['topic_15'].sort_values(ascending=False)[:20]

@lemmatized  покемон     0.268106
             игра        0.109189
             pokemon     0.095733
             nintendo    0.059366
             игрок       0.044063
             пикача      0.039032
             серия       0.038374
             тренер      0.034944
             game        0.033527
             эш          0.030722
             red         0.028671
             аниме       0.025636
             blue        0.024596
             сериал      0.024576
             бой         0.023642
             boy         0.022706
             регион      0.020267
             ruby        0.020025
             sapphire    0.019915
             игровой     0.018577
Name: topic_15, dtype: float64

In [68]:
top_words['topic_1']

[['person', 0.044235989853675974],
 ['игра', 0.0387382873759443],
 ['фильм', 0.034460608860536146],
 ['год', 0.021440806375940213],
 ['персонаж', 0.018763569372780647],
 ['роль', 0.01691528654003225],
 ['время', 0.013179582065132664],
 ['сцена', 0.012967765359657869],
 ['герой', 0.012617957383598114],
 ['актёр', 0.01202547379667657],
 ['серия', 0.01188295046266246],
 ['режиссёр', 0.01181331044036697],
 ['игрок', 0.011805007576893822],
 ['стать', 0.011129362726140676],
 ['сериал', 0.011006356914471157],
 ['картина', 0.01080528813709529],
 ['хороший', 0.010352734596734994],
 ['игровой', 0.009919317386098348],
 ['эпизод', 0.009715262708623107],
 ['версия', 0.009560538600799823]]

In [123]:
phi['topic_15'].sort_values(ascending=False)[:20]

modality     token   
@lemmatized  покемон     0.043134
             игра        0.017567
             pokemon     0.015402
             nintendo    0.009551
             игрок       0.007089
             пикача      0.006280
             серия       0.006174
             тренер      0.005622
             game        0.005394
             эш          0.004943
             red         0.004613
             аниме       0.004124
             blue        0.003957
             сериал      0.003954
             бой         0.003804
             boy         0.003653
             регион      0.003261
             ruby        0.003222
             sapphire    0.003204
             игровой     0.002989
Name: topic_15, dtype: float64

In [117]:
phi.shape, phi0.shape

((264943, 17), (264925, 17))

In [118]:
phi.columns

Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16'],
      dtype='object')

In [119]:
phi0.columns

Index(['background_1', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4',
       'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10',
       'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15'],
      dtype='object')

In [70]:
seed_load_folder

'/data_mil/shared/CompressaAI/BERTopic/results/ruwikigood/0'

In [71]:
! head -n 2 '/data_mil/shared/CompressaAI/BERTopic/results/ruwikigood/0/phi.csv'

,background_1,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13,topic_14,topic_15
aa,1.880537854513533e-05,3.155666363779187e-05,6.524287003967717e-05,0.0,0.0,1.5913009450918555e-05,0.0,1.9111301643428782e-05,0.0,0.0,0.0,0.0,0.0076266546732638605,0.0,0.0,0.0,0.0


In [84]:
phi0.shape, len(top_words)

((264925, 17), 17)

In [85]:
phi0['topic_15'].sort_values(ascending=False)[:20]

@lemmatized  покемон     0.268106
             игра        0.109189
             pokemon     0.095733
             nintendo    0.059366
             игрок       0.044063
             пикача      0.039032
             серия       0.038374
             тренер      0.034944
             game        0.033527
             эш          0.030722
             red         0.028671
             аниме       0.025636
             blue        0.024596
             сериал      0.024576
             бой         0.023642
             boy         0.022706
             регион      0.020267
             ruby        0.020025
             sapphire    0.019915
             игровой     0.018577
Name: topic_15, dtype: float64

In [88]:
top_words.keys()

dict_keys(['background_1', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15'])

In [86]:
top_words['topic_16']

KeyError: 'topic_16'

In [81]:
len(top_words)

17

In [82]:
phi0['topic_14'].sort_values(ascending=False)[:20]

@lemmatized  малдёр           0.138623
             эпизод           0.116320
             агент            0.049091
             person           0.047593
             сериал           0.041120
             тумс             0.037865
             секретный        0.036518
             картер           0.029176
             фбр              0.027501
             материал         0.027373
             серия            0.025630
             fox              0.024078
             сцена            0.023637
             съёмка           0.021304
             сценарий         0.019872
             малдрать         0.019669
             домохозяйство    0.019580
             премьера         0.018778
             сезон            0.018749
             расследовать     0.017247
Name: topic_14, dtype: float64

In [83]:
top_words['topic_15']

[['малдёр', 0.1386228369280603],
 ['эпизод', 0.11631993906668163],
 ['агент', 0.049090525093543905],
 ['person', 0.047593315648508897],
 ['сериал', 0.04111972980469913],
 ['тумс', 0.03786522487257441],
 ['секретный', 0.03651821212407736],
 ['картер', 0.02917647118432673],
 ['фбр', 0.027500518993828337],
 ['материал', 0.027373011451345342],
 ['серия', 0.025629757135196673],
 ['fox', 0.024077945281126726],
 ['сцена', 0.023637398476370437],
 ['съёмка', 0.021304282715615113],
 ['сценарий', 0.019872107173757463],
 ['малдрать', 0.019668540165023844],
 ['домохозяйство', 0.019580396028063024],
 ['премьера', 0.01877765221036062],
 ['сезон', 0.018749264086400926],
 ['расследовать', 0.01724706793607795]]

In [98]:
phi0.columns

Index(['background_1', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4',
       'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10',
       'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15'],
      dtype='object')

In [105]:
phi0['topic_15'].sort_values(ascending=False)[:20]

@lemmatized  покемон     0.268106
             игра        0.109189
             pokemon     0.095733
             nintendo    0.059366
             игрок       0.044063
             пикача      0.039032
             серия       0.038374
             тренер      0.034944
             game        0.033527
             эш          0.030722
             red         0.028671
             аниме       0.025636
             blue        0.024596
             сериал      0.024576
             бой         0.023642
             boy         0.022706
             регион      0.020267
             ruby        0.020025
             sapphire    0.019915
             игровой     0.018577
Name: topic_15, dtype: float64

In [103]:
top_words['topic_15']

[['малдёр', 0.1386228369280603],
 ['эпизод', 0.11631993906668163],
 ['агент', 0.049090525093543905],
 ['person', 0.047593315648508897],
 ['сериал', 0.04111972980469913],
 ['тумс', 0.03786522487257441],
 ['секретный', 0.03651821212407736],
 ['картер', 0.02917647118432673],
 ['фбр', 0.027500518993828337],
 ['материал', 0.027373011451345342],
 ['серия', 0.025629757135196673],
 ['fox', 0.024077945281126726],
 ['сцена', 0.023637398476370437],
 ['съёмка', 0.021304282715615113],
 ['сценарий', 0.019872107173757463],
 ['малдрать', 0.019668540165023844],
 ['домохозяйство', 0.019580396028063024],
 ['премьера', 0.01877765221036062],
 ['сезон', 0.018749264086400926],
 ['расследовать', 0.01724706793607795]]

In [104]:
phi0.columns

Index(['background_1', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4',
       'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10',
       'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15'],
      dtype='object')

In [101]:
top_words.keys()

dict_keys(['background_1', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15'])

In [106]:
top_words

{'background_1': False,
 'topic_0': [['person', 0.0422573407299807],
  ['год', 0.029529524372319878],
  ['время', 0.012102291687147673],
  ['стать', 0.010141119486329416],
  ['город', 0.009177191305420053],
  ['человек', 0.0085192620682621],
  ['часть', 0.008051808916342945],
  ['век', 0.007805910871055686],
  ['имя', 0.007420150137652238],
  ['армия', 0.007204607579826462],
  ['являться', 0.007019571092204588],
  ['мочь', 0.006865394327750571],
  ['иметь', 0.006847778180194606],
  ['война', 0.006679773966025352],
  ['получить', 0.006672689870839094],
  ['работа', 0.006517804032917204],
  ['большой', 0.006191131955561368],
  ['находиться', 0.005972721385662755],
  ['язык', 0.005945349787161841],
  ['книга', 0.005661034032538145]],
 'topic_1': [['person', 0.044235989853675974],
  ['игра', 0.0387382873759443],
  ['фильм', 0.034460608860536146],
  ['год', 0.021440806375940213],
  ['персонаж', 0.018763569372780647],
  ['роль', 0.01691528654003225],
  ['время', 0.013179582065132664],
  ['сц